In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent
processed_dir = project_root / "data" / "processed"

products = pd.read_csv(processed_dir / "products_clean_final.csv")
reviews = pd.read_csv(processed_dir / "reviews_clean_final.csv")
ingredients = pd.read_csv(processed_dir / "ingredients_clean_final.csv")

print(products.shape, reviews.shape, ingredients.shape)

C:\Users\ujjwa\AppData\Local\Temp\ipykernel_12632\1177575670.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv(processed_dir / "reviews_clean_final.csv")


(8494, 27) (1088891, 18) (30, 10)


product intelligence features

price normalization

In [2]:
products["price_usd"]=pd.to_numeric(products["price_usd"],errors="coerce")

products["log_price"]=np.log1p(products["price_usd"])

discount features

In [3]:
products["discount_amount"]=products["value_price_usd"]-products["sale_price_usd"]

products["discount_pct"]=(
    products["discount_amount"]/products["value_price_usd"]
)* 100

product popularity score

In [4]:
products["popularity_score"] = (
    np.log1p(products["loves_count"]) * 0.6 +
    np.log1p(products["reviews"]) * 0.4
)

In [5]:
category_cols = [
    "primary_category",
    "secondary_category",
    "tertiary_category"
]

for col in category_cols:
    products[col] = products[col].fillna("Unknown")

In [6]:
reviews["rating"] = pd.to_numeric(reviews["rating"], errors="coerce")

In [7]:
reviews["positive_signal"] = (
    reviews["rating"] >= 4
).astype(int)

In [8]:
reviews["engagement_score"] = (
    reviews["total_pos_feedback_count"] - reviews["total_neg_feedback_count"]
)

In [9]:
reviews["review_length"] = reviews["review_text"].fillna("").apply(len)

User Level Features

In [10]:
user_features = reviews.groupby("author_id").agg({
    "rating": ["mean", "count"],
    "engagement_score": "mean",
    "positive_signal": "mean"
})

user_features.columns = [
    "avg_rating",
    "review_count",
    "avg_engagement",
    "positive_rate"
]

user_features = user_features.reset_index()

In [11]:
product_features = reviews.groupby("product_id").agg({
    "rating": ["mean", "count"],
    "positive_signal": "mean"
})

product_features.columns = [
    "avg_rating",
    "rating_count",
    "positive_rate"
]

product_features = product_features.reset_index()

Time Based Features

In [12]:
reviews["submission_time"] = pd.to_datetime(reviews["submission_time"])

In [13]:
max_date = reviews["submission_time"].max()

reviews["recency_days"] = (max_date - reviews["submission_time"]).dt.days

In [14]:
reviews["time_weight"] = 1 / (1 + reviews["recency_days"])

Product text features

In [15]:
products["text_blob"] = (
    products["product_name"].fillna("") + " " +
    products["brand_name"].fillna("") + " " +
    products["highlights"].fillna("") + " " +
    products["ingredients"].fillna("")
)

In [16]:
ingredients["ingredient"] = ingredients["ingredient"].fillna("")

ingredient_complexity = ingredients.groupby("ingredient").agg({
    "vegan": "first",
    "pregnancy_safe": "first"
}).reset_index()

In [17]:
ingredients["risk_flag"] = (
    ingredients["avoid_with"].notna() &
    (ingredients["avoid_with"] != "")
).astype(int)

In [18]:
ingredient_score = ingredients.groupby("ingredient").agg({
    "risk_flag": "mean"
}).reset_index()

In [19]:
product_features = reviews.groupby("product_id").agg({
    "rating": ["mean", "count", "std"],
    "total_feedback_count": "mean",
    "positive_signal": "mean",
    "time_weight": "mean"
})

product_features.columns = [
    "avg_rating",
    "rating_count",
    "rating_std",
    "avg_feedback",
    "positive_rate",
    "recency_weight"
]

product_features = product_features.reset_index()

In [20]:
user_features = reviews.groupby("author_id").agg({
    "rating": ["mean", "count", "std"],
    "engagement_score": "mean",
    "positive_signal": "mean",
    "time_weight": "mean"
})

user_features.columns = [
    "avg_rating",
    "review_count",
    "rating_std",
    "avg_engagement",
    "positive_rate",
    "recency_preference"
]

user_features = user_features.reset_index()

In [21]:
reviews["interaction_strength"] = (
    reviews["rating"] *
    reviews["time_weight"] *
    (1 + reviews["total_pos_feedback_count"])
)

In [26]:
# %%
feature_dir = processed_dir / "feature_engineered"
feature_dir.mkdir(parents=True, exist_ok=True)

In [27]:
# %%
products.to_csv(
    feature_dir / "products_features.csv",
    index=False
)

reviews.to_csv(
    feature_dir / "reviews_features.csv",
    index=False
)

user_features.to_csv(
    feature_dir / "user_features.csv",
    index=False
)

product_features.to_csv(
    feature_dir / "product_features.csv",
    index=False
)

ingredient_score.to_csv(
    feature_dir / "ingredient_scores.csv",
    index=False
)

ingredient_complexity.to_csv(
    feature_dir / "ingredient_complexity.csv",
    index=False
)

In [ ]:
# %%
print("Saved files:\n")

for file in feature_dir.glob("*.csv"):
    print(file.name)

Saved files:

ingredient_complexity.csv
ingredient_scores.csv
products_features.csv
product_features.csv
reviews_features.csv
user_features.csv


: 